In [7]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc, Wedge, Polygon
from matplotlib.lines import Line2D
from PIL import Image

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)
OUT_FILE = OUT_DIR / "target_hud_redesigned.gif"

SIZE_PX = 768
DPI = 120
FPS = 24
DURATION_SEC = 5
FRAMES = FPS * DURATION_SEC

HUD = "#36f6ff"
HUD_SOFT = "#7dfcff"
TEXT = "#b7ffff"

rng = np.random.default_rng(12)


def smooth_cycle(t):
    # 0 -> 1 -> 0, без скачка в конце GIF
    return 0.5 - 0.5 * np.cos(2 * np.pi * t)


def add_line(ax, x1, y1, x2, y2, lw=1.2, alpha=0.8):
    ax.add_line(
        Line2D(
            [x1, x2],
            [y1, y2],
            color=HUD,
            linewidth=lw,
            alpha=alpha,
            solid_capstyle="round",
        )
    )


def draw_frame(frame):
    t = frame / FRAMES
    phase = 2 * np.pi * t
    fill = smooth_cycle(t)
    sweep = 360 * fill
    alpha = fill ** 0.45 if fill > 0 else 0

    fig, ax = plt.subplots(figsize=(SIZE_PX / DPI, SIZE_PX / DPI), dpi=DPI)
    fig.patch.set_alpha(0)
    ax.set_facecolor((0, 0, 0, 0))
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect("equal")
    ax.axis("off")

    # ---- outer technical frame ----

    for r, lw, a in [
        (0.94, 1.1, 0.18),
        (0.78, 1.0, 0.24),
        (0.54, 0.9, 0.20),
        (0.20, 1.0, 0.45),
    ]:
        ax.add_patch(
            Circle(
                (0, 0),
                r,
                fill=False,
                edgecolor=HUD,
                linewidth=lw,
                alpha=a,
            )
        )

    # ---- growing arcs ----

    arc_specs = [
        (0.96, 0, 2.8, 0.95),
        (0.82, 60, 2.0, 0.65),
        (0.66, -45, 1.6, 0.55),
    ]

    wobble = 8 * np.sin(phase * 2)

    for radius, offset, lw, base_alpha in arc_specs:
        ax.add_patch(
            Arc(
                (0, 0),
                2 * radius,
                2 * radius,
                angle=wobble,
                theta1=offset - sweep / 2,
                theta2=offset + sweep / 2,
                color=HUD,
                linewidth=lw,
                alpha=base_alpha * alpha,
            )
        )

    # ---- inner segmented ring ----

    seg_count = 36
    visible = int(seg_count * fill)

    for i in range(visible):
        a0 = i * 360 / seg_count
        a1 = a0 + 360 / seg_count * 0.45

        ax.add_patch(
            Wedge(
                (0, 0),
                0.405,
                a0,
                a1,
                width=0.045,
                facecolor=HUD,
                edgecolor="none",
                alpha=0.38 * alpha,
            )
        )

    # ---- four angular brackets / lock frame ----

    bracket_r1 = 0.30
    bracket_r2 = 0.48
    bracket_gap = 0.12

    for base in [45, 135, 225, 315]:
        a = np.deg2rad(base + 6 * np.sin(phase))
        da = np.deg2rad(bracket_gap * 90)

        p1 = np.array([bracket_r1 * np.cos(a - da), bracket_r1 * np.sin(a - da)])
        p2 = np.array([bracket_r2 * np.cos(a), bracket_r2 * np.sin(a)])
        p3 = np.array([bracket_r1 * np.cos(a + da), bracket_r1 * np.sin(a + da)])

        ax.add_patch(
            Polygon(
                [p1, p2, p3],
                closed=False,
                fill=False,
                edgecolor=HUD_SOFT,
                linewidth=1.4,
                alpha=0.35 + 0.45 * fill,
            )
        )

    # ---- central cross, not a grid ----

    pulse = 0.45 + 0.35 * np.sin(phase * 4) ** 2

    add_line(ax, -0.23, 0, -0.07, 0, lw=1.4, alpha=pulse)
    add_line(ax,  0.07, 0,  0.23, 0, lw=1.4, alpha=pulse)
    add_line(ax, 0, -0.23, 0, -0.07, lw=1.4, alpha=pulse)
    add_line(ax, 0,  0.07, 0,  0.23, lw=1.4, alpha=pulse)

    ax.add_patch(
        Circle(
            (0, 0),
            0.045,
            fill=False,
            edgecolor=HUD_SOFT,
            linewidth=1.1,
            alpha=0.8,
        )
    )

    # ---- small technical side ticks ----

    for a_deg in range(0, 360, 30):
        a = np.deg2rad(a_deg)
        r1 = 0.88
        r2 = 0.92 if a_deg % 90 else 0.98

        add_line(
            ax,
            r1 * np.cos(a),
            r1 * np.sin(a),
            r2 * np.cos(a),
            r2 * np.sin(a),
            lw=1.0,
            alpha=0.42,
        )

    # ---- readable telemetry ----

    az = 184.0 + 6.0 * np.sin(phase)
    el = 27.0 + 4.0 * np.sin(phase * 2)
    sig = 72 + int(18 * np.sin(phase * 5))
    dst = 1842 + int(90 * np.sin(phase * 3))

    number_a = "".join(str(rng.integers(0, 10)) for _ in range(10))
    number_b = "".join(str(rng.integers(0, 10)) for _ in range(12))

    ax.text(
        -0.95,
        1.02,
        f"AZ {az:06.2f}   EL {el:05.2f}",
        color=TEXT,
        fontsize=10,
        family="monospace",
        alpha=0.88,
    )

    ax.text(
        0.58,
        -0.96,
        f"DST {dst:04d}\nSIG {sig:02d}%\nSCAN OPT",
        color=TEXT,
        fontsize=9,
        family="monospace",
        alpha=0.88,
    )

    ax.text(
        0,
        -1.13,
        "TARGET LOCK",
        color=TEXT,
        fontsize=12,
        family="monospace",
        ha="center",
        alpha=0.25 + 0.7 * fill,
    )

    ax.text(
        0,
        -1.23,
        f"{number_a}  {number_b}",
        color=TEXT,
        fontsize=10,
        family="monospace",
        ha="center",
        alpha=0.82,
    )

    fig.canvas.draw()

    image = np.asarray(fig.canvas.buffer_rgba())
    pil = Image.fromarray(image).convert("RGBA")

    plt.close(fig)

    return pil


frames = [draw_frame(i) for i in range(FRAMES)]

frames[0].save(
    OUT_FILE,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
    transparency=0,
)

print(f"Saved: {OUT_FILE}")

Saved: animations/target_hud_redesigned.gif
